# Proyecto Final de Probabilidad
## Distribucion Binomial — Efectividad en Penaltis de la Premier League

| | |
|---|---|
| **Grupo** | N. 1 |
| **Integrantes** | Henry Pazmino, Jodie Parrales, Cindy Ayovi, Cesar Gonzales, Mayra Vera |
| **Curso** | 3SA — Computacion en Linea |
| **Docente** | Eco. Jose Julian Coronel Reyes, MSc |
| **Dataset** | Transfermarkt — 21 temporadas (2005/06 a 2025/26) |
| **Observaciones** | 624 jugador-temporada |

---
### Que hace este notebook
Analiza si la efectividad de los penaltis en la Premier League sigue una **distribucion Binomial**.

**En criollo:** queremos saber si el numero de goles de penalti que mete un jugador
se puede predecir con una formula estadistica llamada Binomial. Si se puede,
entonces entrenadores y analistas pueden usar esa formula para tomar decisiones.

Ejecuta cada celda con **Shift+Enter**.

---
## 1. Carga de datos

El dataset contiene **624 filas**. Cada fila representa un jugador en una temporada
especifica. Por ejemplo: Frank Lampard en la temporada 2005/06 lanzo 4 penaltis
y convirtio los 4.

Columnas:
- `season`: temporada (ej. 2005/6)
- `player`: nombre del jugador
- `penalties_taken`: n = cuantos penaltis lanzo ese jugador esa temporada
- `penalties_scored`: k = cuantos de esos penaltis fueron GOL (exitos)

In [ ]:
library(knitr)
url <- "https://raw.githubusercontent.com/josuepaz80-usitee/proyecto-binomial-premier/main/dataset_penaltis_premier.csv"
datos <- read.csv(url)
cat(sprintf("Dataset cargado: %d filas x %d columnas\n\n", nrow(datos), ncol(datos)))
kable(head(datos, 10), caption = "Primeras 10 observaciones", align = "lccc")

---
## 2. Estadisticas descriptivas

Aqui calculamos **p.hat** (p con sombrero), que es la efectividad global:
de todos los penaltis lanzados en 21 temporadas, que porcentaje fue gol.

**Formula:** p.hat = total de goles / total de penaltis lanzados

Este numero sera el parametro `p` de nuestra distribucion Binomial.

In [ ]:
total_n <- sum(datos$penalties_taken)
total_k <- sum(datos$penalties_scored)
p_hat   <- total_k / total_n
est <- data.frame(
  Indicador = c("Penaltis lanzados","Penaltis convertidos","Penaltis fallados","Jugador-temporada","Efectividad global (p.hat)"),
  Valor = c(total_n, total_k, total_n - total_k, nrow(datos), sprintf('%.2f %%', p_hat * 100))
)
kable(est, caption = "Estadisticos globales — Premier League 2005-2026", align = "lr")
cat(sprintf("\nEfectividad global: %.1f%%. p.hat = %.4f\n", p_hat * 100, p_hat))

**Interpretacion:**
- De 1378 penaltis lanzados en 21 temporadas, 1113 fueron gol.
- Solo 265 se fallaron. Eso es un **80.8% de efectividad**.
- En criollo: 8 de cada 10 penaltis en la Premier League terminan en gol.
- Ese 80.8% es nuestro `p.hat`. Lo usamos para predecir cuantos goles
  deberia meter un jugador si lanza N penaltis.

Ahora veamos como se distribuyen los penaltis lanzados (n) y convertidos (k):

In [ ]:
tn <- as.data.frame(table(datos$penalties_taken))
names(tn) <- c("n", "Jugadores")
tn$Porcentaje <- sprintf("%.1f %%", as.numeric(tn$Jugadores) / nrow(datos) * 100)
kable(tn, caption = "Distribucion de n — penaltis lanzados por jugador")

tk <- as.data.frame(table(datos$penalties_scored))
names(tk) <- c("k", "Jugadores")
tk$Porcentaje <- sprintf("%.1f %%", as.numeric(tk$Jugadores) / nrow(datos) * 100)
kable(tk, caption = "Distribucion de k — penaltis convertidos por jugador")
cat(sprintf("\nPromedio: %.1f goles. Desviacion: %.2f\n", mean(datos$penalties_scored), sd(datos$penalties_scored)))

**Interpretacion:**
- **Casi la mitad de los jugadores (48.9%) lanza 1 solo penalti por temporada.**
- Muy pocos lanzan 5 o mas. Esto tiene logica: los penaltis los tiran
  los especialistas del equipo, no cualquiera.
- El **41% de los jugadores convierte exactamente 1 gol** de penalti por temporada.
- En promedio, un jugador convierte **1.8 penaltis por temporada**.
- La mayoria convierte entre 1 y 2. Muy pocos meten 5 o mas.

---
## 3. Condiciones del modelo Binomial

Para que podamos usar la distribucion Binomial, los datos deben cumplir
**4 condiciones**. Si falta una, el modelo no es valido.

Pensalo como los requisitos para entrar a un equipo de futbol:
si no cumplis aunque sea uno, no quedas.

In [ ]:
condiciones <- data.frame(
  Condicion = c("1. Ensayos fijos (n)","2. Dos resultados","3. Independencia","4. Probabilidad constante (p)"),
  Descripcion = c(
    sprintf("n entre %d y %d penaltis por jugador-temporada. Fijo por observacion.", min(datos$penalties_taken), max(datos$penalties_taken)),
    "GOL (exito) o NO GOL (fracaso). Sin resultados intermedios.",
    "Un penalti no afecta al siguiente: distintos jugadores, partidos, temporadas.",
    sprintf("p.hat = %.4f (%.1f%%). Probabilidad base compartida por todos.", p_hat, p_hat * 100)
  ),
  Cumple = c("SI", "SI", "SI", "SI")
)
kable(condiciones, caption = "Verificacion de los 4 supuestos del modelo Binomial")
cat("\nLas 4 condiciones se verifican. Procedemos con Chi-cuadrado.\n")

**Interpretacion:**
- **Condicion 1:** cada jugador tiene un numero fijo de penaltis (n).
  No es que unos lanzan 3 y otros "los que quieran". n esta definido.
- **Condicion 2:** solo hay dos resultados: gol o no gol. No hay "casi gol".
- **Condicion 3:** un penalti no influye en el siguiente. Son independientes.
  Distintos jugadores, distintos partidos, distintas temporadas.
- **Condicion 4:** asumimos que todos los jugadores tienen la misma
  probabilidad base de convertir (80.8%). Esto es una simplificacion:
  sabemos que hay jugadores mejores y peores, pero a nivel de modelo
  estadistico, este supuesto es razonable para la mayoria.

Las 4 condiciones se cumplen. Podemos seguir.

---
## 4. Frecuencias observadas por valor de n

Agrupamos a los jugadores segun cuantos penaltis lanzaron (n = 1, 2, 3, 4, 5).
Para cada grupo, contamos cuantos jugadores convirtieron 0, 1, 2... goles.

**Ejemplo:** de los 73 jugadores que lanzaron 3 penaltis (n=3):
- 0 metieron 0 goles (nadie fallo los 3)
- 5 metieron 1 gol
- 26 metieron 2 goles
- 42 metieron los 3 goles

Estas son las **frecuencias observadas**. Despues las comparamos
con lo que PREDICE la Binomial (frecuencias esperadas).

In [ ]:
valores_n <- sort(unique(datos$penalties_taken))
for (n_val in valores_n[valores_n <= 5]) {
  sub <- subset(datos, penalties_taken == n_val)
  n_jug <- nrow(sub)
  if (n_jug < 10) next
  obs <- as.numeric(table(factor(sub$penalties_scored, levels = 0:n_val)))
  cat(sprintf("\n### n = %d (%d jugadores)\n", n_val, n_jug))
  df <- data.frame(k = 0:n_val, Observado = obs, Porcentaje = sprintf('%.1f %%', obs / n_jug * 100))
  print(kable(df, caption = sprintf("Frecuencias observadas — n = %d", n_val)))
}

---
## 5. Prueba Chi-cuadrado de bondad de ajuste

### Que es el Chi-cuadrado
Es una prueba estadistica que compara dos cosas:
- Lo que **realmente paso** en la Premier League (frecuencias observadas)
- Lo que **deberia haber pasado** si los penaltis siguieran una Binomial perfecta (frecuencias esperadas)

**Si las diferencias son pequenas** (p-valor > 0.05) -> los datos SI siguen Binomial.
**Si las diferencias son grandes** (p-valor <= 0.05) -> los datos NO siguen Binomial.

### Hipotesis
- **H0:** Los datos siguen una distribucion Binomial(n, p.hat)
- **H1:** Los datos NO siguen una distribucion Binomial(n, p.hat)
- **alfa = 0.05** (5% de margen de error)

### Importante
Analizamos cada n por separado porque cada grupo tiene su propio numero
de ensayos. No es lo mismo un jugador que lanzo 1 penalti que uno que lanzo 4.

In [ ]:
chi_por_n <- function(n_val, datos, p_hat) {
  sub <- subset(datos, penalties_taken == n_val)
  n_jug <- nrow(sub)
  if (n_jug < 30) {
    cat(sprintf("\n*** n = %d: solo %d jugadores (< 30).\n", n_val, n_jug))
    cat("    Muestra insuficiente para Chi-cuadrado fiable.\n")
    cat("    Este grupo se EXCLUYE del analisis.\n")
    return(NULL)
  }
  obs_raw <- as.numeric(table(factor(sub$penalties_scored, levels = 0:n_val)))
  esp_prob <- dbinom(0:n_val, size = n_val, prob = p_hat)
  esp_raw <- esp_prob * n_jug
  obs_ag <- c(); esp_ag <- c(); os <- 0; es <- 0
  for (i in 1:length(obs_raw)) {
    os <- os + obs_raw[i]; es <- es + esp_raw[i]
    if (es >= 5 || i == length(obs_raw)) { obs_ag <- c(obs_ag, os); esp_ag <- c(esp_ag, es); os <- 0; es <- 0 }
  }
  if (length(obs_ag) < 2) return(NULL)
  chi <- chisq.test(x = obs_ag, p = esp_ag / sum(esp_ag))
  cat(sprintf("\n>>> n = %d (%d jugadores) >>>\n", n_val, n_jug))
  df <- data.frame(k = 0:n_val, Observado = obs_raw, Esperado = round(esp_raw, 2), Diferencia = round(obs_raw - esp_raw, 2))
  print(kable(df, caption = sprintf("Obs vs Esp — Binomial(n=%d, p=%.4f)", n_val, p_hat)))
  cat(sprintf("\nChi2 = %.4f  |  GL = %d  |  p = %.6f\n", unname(chi$statistic), chi$parameter, chi$p.value))
  if (chi$p.value > 0.05) {
    cat(">>> NO se rechaza H0 — Datos SI siguen Binomial\n")
  } else {
    cat(">>> Se rechaza H0 — Datos NO siguen Binomial\n")
  }
  return(list(n=n_val, n_jug=n_jug, chi2=as.numeric(chi$statistic), gl=as.numeric(chi$parameter), p=chi$p.value))
}
resultados <- list()
for (n_val in 1:5) { res <- chi_por_n(n_val, datos, p_hat); if (!is.null(res)) resultados[[as.character(n_val)]] <- res }
n_ok_global <- sum(sapply(resultados, function(r) r$p > 0.05))
n_t_global <- length(resultados)

**Interpretacion de los resultados del Chi-cuadrado:**

- **n = 1 (305 jugadores): p = 0.000001**
  La Binomial NO ajusta bien. Hay mas jugadores que fallan su unico penalti
  de lo que la Binomial predice (92 observados vs 59 esperados).
  Esto tiene sentido: si un jugador tira 1 solo penalti en toda la temporada,
  probablemente NO es el especialista del equipo.

- **n = 2 (137 jugadores): p = 0.33**
  La Binomial SI ajusta bien. Las diferencias entre observado y esperado
  son pequenas y pueden deberse al azar.

- **n = 3 (73 jugadores): p = 0.60**
  Excelente ajuste. Este es el grupo mas importante porque n = 3 es
  el valor mas comun entre los lanzadores frecuentes.

- **n = 4 (44 jugadores): p = 0.16**
  La Binomial SI ajusta. Aunque la muestra es mas pequena,
  las diferencias no son estadisticamente significativas.

- **n = 5 (28 jugadores): EXCLUIDO**
  Solo 28 jugadores en 21 temporadas lanzaron exactamente 5 penaltis.
  Con tan pocos datos, el Chi-cuadrado no es confiable.
  Es como querer sacar conclusiones de una encuesta de 3 personas.

---
## 6. Tabla resumen de todas las pruebas

Una vista consolidada de los 4 grupos analizados:

In [ ]:
cat("\n===========================================\n")
cat("   RESUMEN — PRUEBAS CHI-CUADRADO (alfa=0.05)\n")
cat("===========================================\n\n")
df_res <- data.frame(
  n = sapply(resultados, `[[`, 'n'),
  Jugadores = sapply(resultados, `[[`, 'n_jug'),
  Chi2 = round(sapply(resultados, `[[`, 'chi2'), 4),
  GL = sapply(resultados, `[[`, 'gl'),
  p.valor = round(sapply(resultados, `[[`, 'p'), 6),
  Decision = sapply(resultados, function(r) ifelse(r$p>0.05, 'BINOMIAL', 'NO Binomial')),
  check.names = FALSE
)
kable(df_res, caption = "Resumen Chi-cuadrado (alfa = 0.05)")
cat(sprintf("\n%d/%d grupos aceptan H0 (%.0f%%).\n", n_ok_global, n_t_global, n_ok_global/n_t_global*100))
if (n_ok_global/n_t_global >= 0.6) {
  cat("La Binomial describe adecuadamente los penaltis de la Premier League.\n")
} else {
  cat("Ajuste no concluyente. Revisar grupos con pocos datos.\n")
}

**Interpretacion general:**

- **3 de 4 grupos (75%) aceptan la hipotesis nula:** los datos SI siguen
  una distribucion Binomial.
- El unico grupo que falla es n=1 (jugadores que lanzaron 1 solo penalti).
  Esto es esperable: los que tiran 1 penalti no son especialistas.
- Para los lanzadores habituales (n=2, 3, 4), la Binomial funciona muy bien.
- **Conclusion:** la distribucion Binomial es un modelo adecuado para
  describir la efectividad en penaltis de la Premier League.

---
## 7. Efectividad por temporada

Una de las condiciones de la Binomial es que **p sea constante**.
Si la efectividad cambia mucho de un ano a otro, ese supuesto se debilita.
Veamos si la efectividad se ha mantenido estable a lo largo de 21 temporadas.

In [ ]:
tmp <- aggregate(cbind(penalties_scored, penalties_taken) ~ season, data = datos, FUN = sum)
tmp$Efectividad <- round(tmp$penalties_scored / tmp$penalties_taken * 100, 1)
tmp <- tmp[order(tmp$season), ]
kable(tmp, caption = "Efectividad en penaltis por temporada")
cat(sprintf("\nOscila entre %.1f%% y %.1f%%. Promedio: %.1f%%.\n", min(tmp$Efectividad), max(tmp$Efectividad), mean(tmp$Efectividad)))
cat("Sin tendencia clara: el supuesto de p constante es razonable.\n")

**Interpretacion:**
- La efectividad va del **74.2% al 90.8%** segun la temporada.
- El promedio historico es **80.7%**, muy cercano a nuestro p.hat de 80.8%.
- No hay una tendencia clara al alza o a la baja: en 2005/06 fue 77.8%
  y en 2025/26 fue 82.9%. Las variaciones son normales por el azar.
- Esto respalda el supuesto de que **p es razonablemente constante**
  a lo largo del tiempo.

---
## 8. Graficos

### Grafico 1 — Histograma de penaltis convertidos
Muestra cuantos goles de penalti convierte un jugador tipico de la Premier
League en una temporada. La linea roja marca el promedio.

In [ ]:
hist(datos$penalties_scored, breaks = seq(-0.5, max(datos$penalties_scored) + 0.5, by = 1),
     col = "#2E6B4F", border = "white", main = "Penaltis Convertidos por Jugador\nPremier League 2005-2026",
     xlab = "k = Goles", ylab = "Jugador-temporada")
abline(v = mean(datos$penalties_scored), col = "#C0392B", lwd = 2.5, lty = 2)
legend("topright", sprintf("Media = %.2f", mean(datos$penalties_scored)), lty = 2, lwd = 2.5, col = "#C0392B", cex = 0.8, bg = "white")

### Grafico 2 — Observado vs Esperado (n = 3)
Comparamos visualmente lo que **realmente paso** (barras verdes) contra
lo que **predice la Binomial** (barras naranjas) para los 73 jugadores
que lanzaron exactamente 3 penaltis. Si las barras tienen alturas parecidas,
el modelo es bueno.

In [ ]:
n_sel <- 3
sub3 <- subset(datos, penalties_taken == n_sel)
obs3 <- as.numeric(table(factor(sub3$penalties_scored, levels = 0:n_sel)))
esp3 <- dbinom(0:n_sel, n_sel, p_hat) * nrow(sub3)
bp <- barplot(rbind(obs3, esp3), beside = TRUE, names.arg = 0:n_sel,
              col = c("#2E6B4F", "#E8A87C"), border = "white",
              main = sprintf("Observado vs Esperado — n=%d (%d jug.)", n_sel, nrow(sub3)),
              xlab = "k = Goles", ylab = "Jugadores", ylim = c(0, max(obs3, esp3) * 1.25))
text(bp[1,], obs3 + 1, labels = obs3, font = 2, cex = 0.8, col = "#2E6B4F")
text(bp[2,], esp3 + 1, labels = round(esp3, 1), font = 2, cex = 0.8, col = "#C07A4B")
legend("topright", fill = c("#2E6B4F", "#E8A87C"), legend = c("Observado", "Esperado Binomial"), cex = 0.75, bg = "white")
# Tabla numerica
dfc <- data.frame(k = 0:n_sel, Observado = obs3, Esperado = round(esp3, 1), Diferencia = round(obs3 - esp3, 1))
cat("\nComparacion numerica:\n")
print(kable(dfc, caption = sprintf("Comparacion n = %d", n_sel)))

**Interpretacion del grafico:**
- Las barras verdes (datos reales) y naranjas (prediccion Binomial)
  son muy parecidas. La diferencia mas grande es en k=3:
  42 jugadores metieron los 3 penaltis, la Binomial esperaba 38.5.
- Una diferencia de 3.5 jugadores en 73 no es estadisticamente significativa.
- El modelo Binomial captura bien la realidad de los penaltis.

### Grafico 3 — Efectividad por temporada
Linea de tiempo que muestra como ha evolucionado la efectividad
de los penaltis temporada por temporada.

In [ ]:
plot(tmp$Efectividad, type = "o", col = "#2E6B4F", lwd = 2.5, pch = 19, cex = 0.9, xaxt = "n",
     main = "Efectividad en Penaltis por Temporada", xlab = "", ylab = "Efectividad (%)", ylim = c(60, 100))
abline(h = p_hat * 100, col = "#C0392B", lwd = 2, lty = 2)
idx <- seq(1, nrow(tmp), by = 3)
axis(1, at = idx, labels = tmp$season[idx], las = 2, cex.axis = 0.6)
legend("bottomleft", legend = c("Por temporada", sprintf("Promedio: %.1f%%", p_hat*100)),
       col = c("#2E6B4F", "#C0392B"), lty = c(1, 2), lwd = c(2.5, 2), pch = c(19, NA), cex = 0.65, bg = "white")

**Interpretacion:**
- Cada punto verde es la efectividad de esa temporada.
- La linea roja horizontal es el promedio historico (80.8%).
- Las temporadas oscilan alrededor de la linea roja, sin tendencia.
- Esto confirma que **p es estable** a lo largo del tiempo,
  uno de los supuestos fundamentales del modelo Binomial.

---
## 9. Conclusiones

Resumen final de todo el analisis:

In [ ]:
conclusiones <- data.frame(
  Aspecto = c("Variable","Modelo","Parametro","Condiciones","Prueba","Muestra","Resultado"),
  Detalle = c(
    "k = penaltis convertidos (discreta, conteo de exitos)",
    "Binomial(n, p) con n fijo por jugador-temporada",
    sprintf("p.hat = %.4f (%.1f%% de efectividad)", p_hat, p_hat*100),
    "Las 4 condiciones se verifican satisfactoriamente",
    "Chi-cuadrado de bondad de ajuste (alfa = 0.05)",
    sprintf("%d jugador-temporada x 21 temporadas Premier League", nrow(datos)),
    sprintf("%d/%d grupos aceptan H0: datos SI siguen Binomial", n_ok_global, n_t_global)
  )
)
kable(conclusiones, caption = "Resumen del analisis — Grupo 1, 3SA")
cat("\nIMPLICACIONES:\n")
cat(sprintf("  - %.0f de cada 10 penaltis son gol.\n", p_hat*10))
cat("  - La Binomial permite calcular P(k|n) exactas.\n")
cat("  - Un entrenador puede estimar cuantos goles esperar de un jugador\n")
cat("    si lanza N penaltis en una temporada.\n")
cat("  - Aplicable a: seleccion de lanzadores, analisis tactico,\n")
cat("    comparacion de jugadores contra el promedio historico.\n")
cat("  - Limitacion: jugadores elite (como Harry Kane) pueden tener un p\n")
cat("    significativamente mayor al promedio.\n")

---
## 10. Referencias

- Transfermarkt. (2026). Premier League — Penalty statistics. https://www.transfermarkt.us
- Devore, J. L. (2021). Probabilidad y estadistica (9a ed.). Cengage.
- Walpole, R. E. et al. (2021). Probabilidad y estadistica para ingenieros (9a ed.). Pearson.
- Agresti, A. (2023). Categorical Data Analysis (3a ed.). Wiley.

---
Proyecto Final — Grupo 1, 3SA, UAE. Julio 2026.